# Elegir optimizador

[`06a`](06a_entrenamiento_variacional.ipynb) entrenó un QAOA con Evolución Diferencial y [`06b`](06b_optimizadores_pso_qng.ipynb) con PSO y QNG. `train()` admite más optimizadores que esos, todos implementados en Rust y pasados en el mismo argumento `method`. Este notebook entrena el mismo problema de MaxCut con el resto, enseña cómo compararlos de forma justa, cómo encadenar dos, cómo seguir y parar una ejecución, y cómo conectar un optimizador escrito fuera de Polypus.

## El problema

El mismo grafo y el mismo QAOA de una capa que en [`06a`](06a_entrenamiento_variacional.ipynb): tres nodos, dos aristas, un mejor corte posible de 2, y dos parámetros libres, `gamma` y `beta`. `common` reúne los argumentos que comparten todas las ejecuciones de abajo, así que cada llamada solo escribe lo que cambia:

In [ ]:
import numpy as np
import polypus

Param = polypus.Param
edges = [(0, 1), (1, 2)]


def cut_value(bitstring):
    bits = bitstring[::-1]  # bits[i] = valor del qubit i
    return sum(1 for i, j in edges if bits[i] != bits[j])


qc = polypus.Circuit(3)
qc.h(0).h(1).h(2)
qc.rzz(0, 1, Param(0))
qc.rzz(1, 2, Param(0))
qc.rx(0, Param(1)).rx(1, Param(1)).rx(2, Param(1))
qc.measure_all()

common = dict(
    shots=1000,
    n_qpus=1,
    expectation_function=cut_value,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    seed=7,
)

## Cinco optimizadores sobre el mismo problema

- **`DE`** es la Evolución Diferencial de [`06a`](06a_entrenamiento_variacional.ipynb). `strategy="shade"` y `strategy="lshade"` son variantes adaptativas que ajustan los propios parámetros de DE mientras se ejecutan; L-SHADE además reduce la población a medida que avanza.
- **`CMAES`** también mantiene una población, pero la muestrea de una distribución que desplaza y deforma hacia los mejores candidatos.
- **`Adam`** sigue el gradiente del coste, estimado con dos circuitos más por parámetro, con un paso adaptado a cada parámetro.
- **`SPSA`** también sigue el gradiente, pero lo estima entero con solo dos circuitos por iteración, moviendo todos los parámetros a la vez en una dirección aleatoria.

Al compararlos hay una trampa: `best_fitness` es el mejor de muchos valores medidos con ruido de muestreo, así que tiende a salir por encima del valor real de sus parámetros, más cuantos más candidatos prueba un optimizador. `final_shots` vuelve a medir los mejores candidatos al terminar el entrenamiento, con muchos más shots, y lo devuelve como `final_fitness`, que es la cifra justa para comparar:

In [ ]:
optimizers = {
    "DE": polypus.DE(generations=30, population_size=10, seed=7),
    "L-SHADE": polypus.DE(
        strategy="lshade", generations=30, population_size=10, seed=7
    ),
    "CMA-ES": polypus.CMAES(generations=30, seed=7),
    "Adam": polypus.Adam(max_iters=30, seed=7),
    "SPSA": polypus.SPSA(max_iters=100, seed=7),
}
for name, optimizer in optimizers.items():
    result = polypus.train(
        qc,
        optimizer,
        dimensions=qc.num_params,
        id=f"maxcut-{name}",
        final_shots=20000,
        **common,
    )
    print(
        f"{name:>8}: best_fitness {result.best_fitness:.3f}  "
        f"final_fitness {result.final_fitness:.3f}  "
        f"iterations {result.iterations_run}"
    )

En un problema tan pequeño todos los optimizadores llegan al mismo corte, en torno a 1,65, por debajo del 2 que, como se vio en [`06a`](06a_entrenamiento_variacional.ipynb), una sola capa de QAOA no alcanza en este grafo, y las diferencias en `best_fitness` son ruido de muestreo. Las diferencias aparecen en problemas más grandes: la [guía de entrenamiento](https://github.com/Bahia-Software/polypus/blob/main/docs/guide/training.md#choosing-an-optimizer), en inglés, los compara en grafos de 10 nodos con circuitos más profundos, con el mismo presupuesto de evaluaciones de circuito.

## Rotosolve: un parámetro cada vez, de forma exacta

`Rotosolve` no lleva tamaño de paso. Mueve un parámetro cada vez a su mejor valor exacto, calculado con tres circuitos, algo que solo es posible cuando cada parámetro es el ángulo de una única rotación. En este QAOA `Param(0)` es el ángulo de las dos puertas `rzz`, así que Rotosolve rechaza el circuito antes de ejecutar nada:

In [ ]:
try:
    polypus.train(
        qc, polypus.Rotosolve(seed=7), dimensions=qc.num_params, id="roto", **common
    )
except ValueError as e:
    print(e)

Un ansatz eficiente en hardware, una capa de rotaciones `ry` con un parámetro propio cada una, una cadena de `cx` y otra capa de `ry`, sí cumple la condición. A diferencia de la capa de QAOA, este circuito puede preparar por sí mismo la mejor partición, así que Rotosolve llega a un corte cercano al máximo de 2:

In [ ]:
hea = polypus.Circuit(3)
for q in range(3):
    hea.ry(q, Param(q))
hea.cx(0, 1).cx(1, 2)
for q in range(3):
    hea.ry(q, Param(3 + q))
hea.measure_all()

roto = polypus.train(
    hea,
    polypus.Rotosolve(sweeps=5, seed=7),
    dimensions=hea.num_params,
    id="roto-hea",
    **common,
)
print(roto.best_fitness, roto.iterations_run, roto.converged)

## Encadenar dos optimizadores con `initial_point`

`initial_point` hace empezar una ejecución desde unos parámetros dados en vez de desde un punto aleatorio. Eso permite encadenar dos optimizadores: uno que explora todo el espacio, como L-SHADE, y luego uno que refina localmente desde su mejor punto, como Adam. En la comparativa de la guía, dar la mitad del presupuesto a cada uno fue la mejor de todas las opciones:

In [ ]:
explore = polypus.train(
    qc,
    polypus.DE(strategy="lshade", generations=15, population_size=10, seed=7),
    dimensions=qc.num_params,
    id="explore",
    **common,
)
refine = polypus.train(
    qc,
    polypus.Adam(max_iters=15, seed=7),
    dimensions=qc.num_params,
    id="refine",
    initial_point=explore.best_params,
    **common,
)
print("L-SHADE:     ", explore.best_fitness)
print("luego Adam:  ", refine.best_fitness)

## Seguir y parar una ejecución

`callback` es una función que `train()` llama después de cada iteración con un `TrainProgress`: la iteración, el mejor fitness y los mejores parámetros hasta el momento, y los candidatos evaluados. Puede imprimir el progreso, guardar un punto de control o, devolviendo `True`, parar la ejecución, aquí en cuanto el corte medio llega a 1,4. Una ejecución parada devuelve su resultado como siempre, con `converged=False`:

In [ ]:
def informe(progress):
    if progress.iteration % 5 == 0:
        print(
            f"iteración {progress.iteration:>2}: mejor {progress.best_fitness:.3f}, "
            f"{progress.evaluations} evaluaciones"
        )
    return progress.best_fitness >= 1.4


watched = polypus.train(
    qc,
    polypus.Adam(max_iters=60, seed=7),
    dimensions=qc.num_params,
    id="watched",
    callback=informe,
    **common,
)
print(
    "parada en la iteración", watched.iterations_run, "| converged:", watched.converged
)

## Un optimizador de fuera de Polypus: `External`

`polypus.External` ejecuta cualquier optimizador con forma ask/tell a través del mismo planificador que los demás: `ask()` devuelve un lote de candidatos, Polypus evalúa el lote, repartido entre las QPUs, y `tell(candidates, values)` le devuelve su fitness, siempre para maximizar. `max_evaluations` limita los candidatos evaluados. Aquí el optimizador es una pequeña búsqueda local escrita a mano:

In [ ]:
class BusquedaLocal:
    """Muestrea un lote alrededor del mejor punto hasta ahora y se mueve allí si mejora."""

    def __init__(self, dimensions, batch_size, step, seed):
        self.rng = np.random.default_rng(seed)
        self.best = self.rng.uniform(0, np.pi, dimensions)
        self.best_value = -np.inf
        self.batch_size = batch_size
        self.step = step

    def ask(self):
        noise = self.rng.standard_normal((self.batch_size, len(self.best)))
        return self.best + self.step * noise

    def tell(self, candidates, values):
        i = int(np.argmax(values))
        if values[i] > self.best_value:
            self.best, self.best_value = np.array(candidates[i]), values[i]


external = polypus.train(
    qc,
    polypus.External(
        BusquedaLocal(2, batch_size=8, step=0.3, seed=7), max_evaluations=200
    ),
    dimensions=qc.num_params,
    id="external",
    **common,
)
print(external.best_fitness, external.iterations_run, external.converged)

`polypus.ask_tell` trae adaptadores ya hechos para Optuna, Nevergrad y pycma, `from_optuna`, `from_nevergrad` y `from_pycma`, que se instalan con `pip install "polypus-quantum[optuna]"` (o `[nevergrad]`, `[cma]`); la [guía](https://github.com/Bahia-Software/polypus/blob/main/docs/guide/training.md#external-optimizers), en inglés, muestra cada uno.

## Resumen

Este notebook ha entrenado el mismo QAOA con L-SHADE, CMA-ES, Adam y SPSA, los ha comparado con `final_fitness` en vez de con el `best_fitness` ruidoso, ha entrenado un circuito que cumple las condiciones de Rotosolve, ha encadenado L-SHADE y Adam con `initial_point`, ha seguido y parado una ejecución con `callback`, y ha conectado un optimizador propio con `External`.

La siguiente sección entrena modelos con datos mediante `polypus.qml`, el módulo nativo de aprendizaje automático cuántico.

## Siguiente paso

Continuar con: [`07a_clasificador_nativo.ipynb`](07a_clasificador_nativo.ipynb).